In [1]:
import pandas as pd
import numpy as np
import openpyxl as op

In [2]:
df = pd.read_excel('GO IT Final Project.xlsx')
df.head()


,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Tax%,Region,Payment Method
0,10001,2024-01-01,Electronics,iPhone 14 Pro,2.000,"999,99",1999.98,3.0,North America,Credit Card
1,10002,2024-01-02,Home Appliances,Dyson V11 Vacuum,1.000,"499,99",499.99,3.0,Europe,PayPal
2,10003,2024-01-03,Clothing,Levi's 501 Jeans,3.200,"69,99",209.97,3.0,Asia,Debit Card
3,10004,2024-01-04,Books,The Da Vinci Code,4.001,"15,99",63.96,3.0,North America,Credit Card
4,10005,2024-01-05,Beauty Products,Neutrogena Skincare Set,1.000,"89,99",89.99,3.0,Europe,PayPal


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    240 non-null    int64  
 1   Date              240 non-null    str    
 2   Product Category  236 non-null    str    
 3   Product Name      235 non-null    str    
 4   Units Sold        240 non-null    float64
 5   Unit Price        240 non-null    object 
 6   Total Revenue     240 non-null    float64
 7   Tax%              222 non-null    float64
 8   Region            240 non-null    str    
 9   Payment Method    240 non-null    str    
dtypes: float64(3), int64(1), object(1), str(5)
memory usage: 18.9+ KB


In [4]:
null_prod_name = df[df['Product Name'].isnull()]
null_prod_name

,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Tax%,Region,Payment Method
6,10007,2024-01-07,Electronics,NaN,1.000,"2499,99",2499.99,3.0,North America,Credit Card
54,10055,2024-02-24,Electronics,NaN,4.001,"59,99",239.96,5.0,North America,Credit Card
84,10085,2024-03-25,Electronics,NaN,1.000,"99,99",99.99,4.0,North America,Credit Card
162,10163,2024-06-11,Electronics,NaN,1.000,"1199,99",1199.99,222.0,North America,Credit Card
175,10176,2024-06-24,Home Appliances,NaN,2.000,"49,99",99.98,5.0,Europe,PayPal


In [5]:
df['Product Name'] = df['Product Name'].fillna('Unknown Product')
null_prod_name

,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Tax%,Region,Payment Method
6,10007,2024-01-07,Electronics,NaN,1.000,"2499,99",2499.99,3.0,North America,Credit Card
54,10055,2024-02-24,Electronics,NaN,4.001,"59,99",239.96,5.0,North America,Credit Card
84,10085,2024-03-25,Electronics,NaN,1.000,"99,99",99.99,4.0,North America,Credit Card
162,10163,2024-06-11,Electronics,NaN,1.000,"1199,99",1199.99,222.0,North America,Credit Card
175,10176,2024-06-24,Home Appliances,NaN,2.000,"49,99",99.98,5.0,Europe,PayPal


In [6]:
null_prod_cat = df[df['Product Category'].isnull()]
null_prod_cat

,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Tax%,Region,Payment Method
38,10039,2024-02-08,NaN,Columbia Fleece Jacket,4.001,"59,99",239.96,4.0,Asia,Debit Card
113,10114,2024-04-23,NaN,Fitbit Versa 3,3.200,"229,95",689.85,NaN,Asia,Credit Card
161,10162,2024-06-10,NaN,TriggerPoint GRID Foam Roller,2.000,"34,99",69.98,222.0,Asia,Credit Card
194,10195,2024-07-13,NaN,Adidas Ultraboost Running Shoes,2.000,"179,99",359.98,5.0,Asia,Debit Card


In [7]:
mapping = (df.dropna(subset=['Product Category'])
             .drop_duplicates('Product Name')
             .set_index('Product Name')['Product Category'])

mask = df['Product Category'].isnull()
df.loc[mask, 'Product Category'] = df.loc[mask, 'Product Name'].map(mapping)

In [8]:
print(sorted(df['Product Category'].dropna().unique()))

['Beauty Products', 'Books', 'Clothing', 'Electronics', 'Home Appliances', 'Sports']


In [9]:
manual = {
    'Columbia Fleece Jacket': 'Clothing',
    'Fitbit Versa 3': 'Electronics',
    'TriggerPoint GRID Foam Roller': 'Sports',
    'Adidas Ultraboost Running Shoes': 'Sports',
}

mask = df['Product Category'].isnull()
df.loc[mask, 'Product Category'] = df.loc[mask, 'Product Name'].map(manual)

print(df['Product Category'].isnull().sum())   # має бути 0

0


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    240 non-null    int64  
 1   Date              240 non-null    str    
 2   Product Category  240 non-null    str    
 3   Product Name      240 non-null    str    
 4   Units Sold        240 non-null    float64
 5   Unit Price        240 non-null    object 
 6   Total Revenue     240 non-null    float64
 7   Tax%              222 non-null    float64
 8   Region            240 non-null    str    
 9   Payment Method    240 non-null    str    
dtypes: float64(3), int64(1), object(1), str(5)
memory usage: 18.9+ KB


In [11]:
df['Transaction ID'] = df['Transaction ID'].astype('str')
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['Product Category'] = df['Product Category'].astype('category')
df['Product Name'] = df['Product Name'].astype('category')
df['Unit Price'] = df['Unit Price'].astype('str').str.replace(',', '.').astype('float')
df['Total Revenue'] = df['Total Revenue'].astype('float')
df['Quantity'] = df['Total Revenue'] / df['Unit Price']
df['Quantity'] = df['Quantity'].astype('int')
df['Region'] = df['Region'].astype('category')
df['Payment Method'] = df['Payment Method'].astype('category')

In [12]:
df = df.drop(columns=['Units Sold'])

In [13]:
df['Tax%'].value_counts()

Tax%
5.0      83
2.0      40
3.0      38
4.0      37
222.0    24
Name: count, dtype: int64

In [14]:
df['Tax%'] = df['Tax%'].replace(222.0, np.nan)   # np.nan, щоб колонка лишилась float
print(df['Tax%'].isna().sum()) 

42


In [15]:
print(pd.crosstab(df['Region'], df['Tax%']))

Tax%           2.0  3.0  4.0  5.0
Region                           
Asia            13   12   13   28
Europe          14   13   12   27
North America   13   13   12   28


In [16]:
print(pd.crosstab(df['Date'].dt.month, df['Tax%']))

Tax%  2.0  3.0  4.0  5.0
Date                    
1       0   25    6    0
2       0    0   14   15
3      16    0   11    1
4       0   13    6    0
5      24    0    0    0
6       0    0    0    9
7       0    0    0   31
8       0    0    0   27


In [17]:
tax_known = df[df['Tax%'].notna()].sort_values('Date')
print(tax_known[['Date', 'Tax%']].drop_duplicates('Date').to_string())

          Date  Tax%
0   2024-01-01   3.0
1   2024-01-02   3.0
2   2024-01-03   3.0
3   2024-01-04   3.0
4   2024-01-05   3.0
5   2024-01-06   3.0
6   2024-01-07   3.0
7   2024-01-08   3.0
8   2024-01-09   3.0
9   2024-01-10   3.0
13  2024-01-11   3.0
14  2024-01-15   3.0
15  2024-01-16   3.0
16  2024-01-17   3.0
18  2024-01-19   3.0
21  2024-01-22   3.0
22  2024-01-23   3.0
23  2024-01-24   3.0
24  2024-01-25   3.0
25  2024-01-26   4.0
26  2024-01-27   4.0
27  2024-01-28   4.0
28  2024-01-29   4.0
29  2024-01-30   4.0
30  2024-01-31   4.0
31  2024-02-01   4.0
32  2024-02-02   4.0
36  2024-02-03   4.0
37  2024-02-07   4.0
38  2024-02-08   4.0
39  2024-02-09   4.0
40  2024-02-10   4.0
41  2024-02-11   4.0
42  2024-02-12   4.0
43  2024-02-13   4.0
46  2024-02-14   5.0
47  2024-02-17   5.0
48  2024-02-18   5.0
49  2024-02-19   5.0
50  2024-02-20   5.0
51  2024-02-21   5.0
52  2024-02-22   5.0
53  2024-02-23   5.0
54  2024-02-24   5.0
55  2024-02-25   5.0
56  2024-02-26   5.0
57  2024-02-2

In [18]:
df = df.sort_values('Date')
df['Tax%'] = df['Tax%'].ffill().bfill()
print(df['Tax%'].isna().sum())

0


In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 240 entries, 0 to 239
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    240 non-null    str           
 1   Date              240 non-null    datetime64[us]
 2   Product Category  240 non-null    category      
 3   Product Name      240 non-null    category      
 4   Unit Price        240 non-null    float64       
 5   Total Revenue     240 non-null    float64       
 6   Tax%              240 non-null    float64       
 7   Region            240 non-null    category      
 8   Payment Method    240 non-null    category      
 9   Quantity          240 non-null    int64         
dtypes: category(4), datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 16.2 KB


In [20]:
df.head()

,Transaction ID,Date,Product Category,Product Name,Unit Price,Total Revenue,Tax%,Region,Payment Method,Quantity
0,10001,2024-01-01,Electronics,iPhone 14 Pro,999.99,1999.98,3.0,North America,Credit Card,2
1,10002,2024-01-02,Home Appliances,Dyson V11 Vacuum,499.99,499.99,3.0,Europe,PayPal,1
2,10003,2024-01-03,Clothing,Levi's 501 Jeans,69.99,209.97,3.0,Asia,Debit Card,3
3,10004,2024-01-04,Books,The Da Vinci Code,15.99,63.96,3.0,North America,Credit Card,4
4,10005,2024-01-05,Beauty Products,Neutrogena Skincare Set,89.99,89.99,3.0,Europe,PayPal,1


In [21]:
df = df.sort_values('Date').reset_index(drop=True)
df.to_excel('GO_IT_cleaned_by_pandas.xlsx', index=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    240 non-null    str           
 1   Date              240 non-null    datetime64[us]
 2   Product Category  240 non-null    category      
 3   Product Name      240 non-null    category      
 4   Unit Price        240 non-null    float64       
 5   Total Revenue     240 non-null    float64       
 6   Tax%              240 non-null    float64       
 7   Region            240 non-null    category      
 8   Payment Method    240 non-null    category      
 9   Quantity          240 non-null    int64         
dtypes: category(4), datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 14.4 KB


In [22]:
dupes = (df.groupby('Product Name')['Product Category']
           .nunique()
           .loc[lambda s: s > 1])
print(dupes)

Product Name
Dyson Supersonic Hair Dryer    2
Garmin Forerunner 945          2
Unknown Product                2
Name: Product Category, dtype: int64


In [23]:
df[df['Product Name'].isin(['Garmin Forerunner 945', 'Dyson Supersonic Hair Dryer'])][
    ['Transaction ID', 'Product Name', 'Product Category', 'Quantity', 'Unit Price']
]

,Transaction ID,Product Name,Product Category,Quantity,Unit Price
16,10017,Dyson Supersonic Hair Dryer,Beauty Products,1,399.99
18,10019,Garmin Forerunner 945,Electronics,2,499.99
127,10128,Dyson Supersonic Hair Dryer,Home Appliances,2,399.99
221,10222,Garmin Forerunner 945,Sports,1,599.99
